<a href="https://colab.research.google.com/github/selinalee09/Python-Works/blob/main/hcp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix

In [3]:
raw = pd.read_csv('1_ Raw data.csv')
dtr = pd.read_csv('2_ DTR_146.csv')
removed = pd.read_csv('3_ Removed species.csv')

In [8]:
dtr_accessions = set(dtr['Protein Accession'])
removed_accessions = set(removed['Protein Accession'].dropna())
print(f"Loaded {len(raw)} total proteins in raw data")
print(f"DTR (difficult to remove) list: {len(dtr_accessions)} proteins")
print(f"Removed/contaminant list: {len(removed_accesions)} proteins")

Loaded 3900 total proteins in raw data
DTR (difficult to remove) list: 146 proteins
Removed/contaminant list: 24 proteins


In [9]:
KD = {'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5, 'Q': -3.5, 'E': -3.5,'G': -0.4,'H': -3.2, 'I': 4.5, 'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8,'P': -1.6, 'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2}
def gravy(seq):
  seq = re.sub(r'[^A-Z]', '', str(seq).upper())
  vals = [KD[a] for a in seq if a in KD]
  return sum(vals) / len(vals) if vals else np.nan
raw['GRAVY'] = raw['Sequence'].apply(gravy)

In [10]:
serotypes = ['AAV2', 'AAV5', 'AAV8', 'AAV9']
rows=[]
for _, protein_row in raw.iterrows():
  accession = protein_row['Protein Accession']
  if accession in removed_accessions:
    continue
  for sero in serotypes:
    col_r1 = f'{sero}_AAVX_R1'
    col_r2 = f'{sero}_AAVX_R2'
    v1 = protein_row.get(col_r1, 0)
    v2 = protein_row.get(col_r2, 0)
    v1 = 0 if pd.isna(v1) else v1
    v2 = 0 if pd.isna(v2) else v2

    detected_early = (v1>0) or (v2 >0)
    if not detected_early:
      continue
    label =1 if accession in dtr_accessions else 0
    rows.append({'accession': accession, 'protein_name': protein_row['Protein name'], 'serotype': sero, 'MW_Da': protein_row['MW (Da)'], 'pI': protein_row['pI'], 'GRAVY': protein_row['GRAVY'], 'survived_purification': label})

df = pd.DataFrame(rows).dropna(subset=['MW_Da', 'pI', 'GRAVY'])
print(f"\nFinal (protein, serotype) dataset: {len(df)} rows")
print(df['serotype'].value_counts())
print(df.groupby('serotype')['survived_purification'].mean())


Final (protein, serotype) dataset: 11047 rows
serotype
AAV9    3541
AAV5    3266
AAV8    2137
AAV2    2103
Name: count, dtype: int64
serotype
AAV2    0.054208
AAV5    0.041029
AAV8    0.056621
AAV9    0.039254
Name: survived_purification, dtype: float64


In [12]:
results = {}
feature_cols = ['MW_Da', 'pI', 'GRAVY']

print("<LEAVE-ONE-SEROTYPE-OUT HOLDOUT TEST>")

for held_out in serotypes:
    train_df = df[df['serotype'] != held_out]
    test_df = df[df['serotype'] == held_out]

    X_train = train_df[feature_cols].values
    y_train = train_df['survived_purification'].values
    X_test = test_df[feature_cols].values
    y_test = test_df['survived_purification'].values

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    model = LogisticRegression(class_weight='balanced', max_iter=1000)
    model.fit(X_train_s, y_train)

    y_proba = model.predict_proba(X_test_s)[:, 1]
    y_pred = model.predict(X_test_s)

    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    results[held_out] = {'AUC': auc, 'precision': prec, 'recall': rec, 'f1': f1}

    print(f"\nHeld out: {held_out}  (train on the other 3 serotypes)")
    print(f"  Test set size: {len(test_df)}  (positives: {y_test.sum()})")
    print(f"  ROC-AUC:   {auc:.3f}")
    print(f"  Precision: {prec:.3f}")
    print(f"  Recall:    {rec:.3f}")
    print(f"  F1:        {f1:.3f}")
    print(f"  Confusion matrix [[TN,FP],[FN,TP]]:\n{cm}")



<LEAVE-ONE-SEROTYPE-OUT HOLDOUT TEST>

Held out: AAV2  (train on the other 3 serotypes)
  Test set size: 2103  (positives: 114)
  ROC-AUC:   0.767
  Precision: 0.130
  Recall:    0.711
  F1:        0.220
  Confusion matrix [[TN,FP],[FN,TP]]:
[[1448  541]
 [  33   81]]

Held out: AAV5  (train on the other 3 serotypes)
  Test set size: 3266  (positives: 134)
  ROC-AUC:   0.777
  Precision: 0.103
  Recall:    0.709
  F1:        0.180
  Confusion matrix [[TN,FP],[FN,TP]]:
[[2305  827]
 [  39   95]]

Held out: AAV8  (train on the other 3 serotypes)
  Test set size: 2137  (positives: 121)
  ROC-AUC:   0.783
  Precision: 0.143
  Recall:    0.711
  F1:        0.238
  Confusion matrix [[TN,FP],[FN,TP]]:
[[1500  516]
 [  35   86]]

Held out: AAV9  (train on the other 3 serotypes)
  Test set size: 3541  (positives: 139)
  ROC-AUC:   0.774
  Precision: 0.093
  Recall:    0.712
  F1:        0.165
  Confusion matrix [[TN,FP],[FN,TP]]:
[[2442  960]
 [  40   99]]


In [13]:
summary = pd.DataFrame(results).T
summary.loc['MEAN'] = summary.mean()
print("<SUMMARY ACROSS ALL 4 HOLDOUTS")
print(summary.round(3))

<SUMMARY ACROSS ALL 4 HOLDOUTS
        AUC  precision  recall     f1
AAV2  0.767      0.130   0.711  0.220
AAV5  0.777      0.103   0.709  0.180
AAV8  0.783      0.143   0.711  0.238
AAV9  0.774      0.093   0.712  0.165
MEAN  0.775      0.117   0.711  0.201


In [15]:
df.to_csv('aav_hcp_by_serotype.csv', index=False)
summary.to_csv('holdout_results_summary.csv')

In [16]:
print("\nSaved: aav_hcp_by_serotype.csv, holdout_results_summary.csv")


Saved: aav_hcp_by_serotype.csv, holdout_results_summary.csv
